In [1]:
import copy
import uuid

import ome_types
import tifffile

In [2]:
from pathlib import Path
import xmltodict
import pandas as pd
import numpy as np
import tifffile as tiff
import xmltodict

import matplotlib.pyplot as plt

In [3]:
raw_data_folder = Path(r"Y:\LUKE\ATLAS-projects\Lenka-TA19-22_data\session_385872141\TA19-100nm\S_001_2044144885")

# Check if files in the folder have the ".ve-tie" extension
tie_file = None
mif_file = None
for file in raw_data_folder.iterdir():  # Iterate over all items in the folder
    if file.is_file() and file.suffix == ".ve-tie":  # Check if it's a file with the desired extension
        print(f"File with '.ve-tie' extension found: {file.name}")
        tie_file = file
    if file.is_file() and file.suffix == ".ve-mif":  # Check if it's a file with the desired extension
        print(f"File with '.ve-mif' extension found: {file.name}")
        mif_file = file

with open(mif_file, "r", encoding="utf-8") as f:
    mif_dict = xmltodict.parse(f.read())
    

File with '.ve-mif' extension found: MosaicInfo_S_001_2044144885.ve-mif
File with '.ve-tie' extension found: MosaicInfo_S_001_2044144885.ve-tie


In [4]:
def extract_tif_metadata(tif_path):
    """
    Extracts metadata from a `.tif` file and returns it as a dictionary.

    Parameters:
    ----------
    tif_path : str or Path
        Path to the `.tif` file.

    Returns:
    -------
    dict
        A dictionary containing the extracted metadata. If the 'FibicsXML' key is
        missing, an empty dictionary is returned instead.
    """
    with tiff.TiffFile(tif_path) as tif:
        metadata = tif.pages[0].tags  # Extract metadata from the first page
        
        # Convert metadata to dictionary
        metadata_dict = {tag.name: tag.value for tag in metadata.values()}

    # Ensure 'FibicsXML' key exists, otherwise return an empty dictionary
    if "FibicsXML" not in metadata_dict:
        print("Warning: 'FibicsXML' metadata not found in the TIFF file.")
        metadata_dict['FibicsDict'] = {}

    # Parse XML into a Python dictionary
    try:
        fibics_dict = xmltodict.parse(metadata_dict['FibicsXML'])
        fibics_dict = fibics_dict.get('Fibics', {})  # Extract 'Fibics' section if it exists
        metadata_dict['FibicsDict'] = fibics_dict
    except Exception as e:
        print(f"Error parsing FibicsXML: {e}")
        metadata_dict['FibicsDict'] = {}  # Return an empty dictionary if parsing fails

    return metadata_dict

def get_pixel_size_from_tif(filename, raw_data_folder):
    """
    Extracts the pixel size (in microns) from the TIFF metadata.

    Parameters:
    ----------
    filename : str
        The TIFF filename from the DataFrame.
    raw_data_folder : Path
        The directory where the TIFF files are stored.

    Returns:
    -------
    float
        The pixel size in microns, or NaN if extraction fails.
    """
    tif_file = raw_data_folder.joinpath(Path(filename).name)

    try:
        # Extract metadata
        tif_metadata = extract_tif_metadata(tif_file)
        fibics_dict = tif_metadata.get('FibicsDict', {})

        # Extract pixel size (Ux) if available
        pix_size_micron = float(fibics_dict['Scan']['Ux'])
        return pix_size_micron

    except (KeyError, ValueError, FileNotFoundError) as e:
        print(f"Warning: Could not extract pixel size for {filename}: {e}")
        return float('nan')  # Return NaN for missing values

def get_image_size_from_tif(filename, raw_data_folder):
    """
    Extracts ImageWidth and ImageHeight from the Fibics metadata in the TIFF file.

    Parameters:
    ----------
    filename : str
        The TIFF filename from the DataFrame.
    raw_data_folder : Path
        The directory where the TIFF files are stored.

    Returns:
    -------
    tuple (int, int)
        The image width and height extracted from the TIFF metadata.
        Returns (NaN, NaN) if extraction fails.
    """
    tif_file = raw_data_folder.joinpath(Path(filename).name)

    try:
        # Extract metadata
        tif_metadata = extract_tif_metadata(tif_file)
        fibics_dict = tif_metadata.get('FibicsDict', {})

        # Extract image width & height if available
        image_width = int(fibics_dict['Image']['Width'])
        image_height = int(fibics_dict['Image']['Height'])

        return image_width, image_height

    except (KeyError, ValueError, FileNotFoundError) as e:
        print(f"Warning: Could not extract image size for {filename}: {e}")
        return np.nan, np.nan  # Return NaN values for missing metadata

def normalize_angle(angle):
    """
    Normalizes an angle to the range [-180, 180] degrees.

    Parameters:
    ----------
    angle : float
        The input angle in degrees.

    Returns:
    -------
    float
        The normalized angle within [-180, 180] range.
    """
    return ((angle + 180) % 360) - 180

def rotate_points(points, angle_degrees):
    """
    Rotates one or multiple 2D points (x, y) by a given angle in degrees.

    Parameters:
    ----------
    points : tuple (x, y) or list of tuples [(x1, y1), (x2, y2), ...]
        The original point(s) to be rotated.
    angle_degrees : float
        The rotation angle in degrees.

    Returns:
    -------
    tuple (x', y') or list of tuples [(x1', y1'), (x2', y2'), ...]
        The rotated point(s).
    """
    # Convert input to NumPy array
    points_array = np.array(points, dtype=np.float64)  # Ensure it's float for precision

    # If a single point was given, reshape it to (1, 2)
    if points_array.ndim == 1:
        points_array = points_array.reshape(1, 2)

    # Convert angle to radians
    angle_radians = np.radians(angle_degrees)

    # Define the rotation matrix
    rotation_matrix = np.array([
        [np.cos(angle_radians), -np.sin(angle_radians)],
        [np.sin(angle_radians),  np.cos(angle_radians)]
    ])

    # Apply rotation (matrix multiplication)
    rotated_points = points_array @ rotation_matrix.T  # Transpose for correct multiplication

    # Convert back to original format (tuple or list of tuples)
    if len(rotated_points) == 1:
        return tuple(rotated_points[0])  # Return a single tuple for single input
    return [tuple(point) for point in rotated_points]  # Return a list of tuples for multiple points

In [5]:
mif_dict['MosaicInfo']['TileInfo'].keys()
# Extract tile list (or single dictionary)
mif_tile_list = mif_dict['MosaicInfo']['Tiles']['Tile']

# Ensure mif_tile_list is always a list
if isinstance(mif_tile_list, dict):  # If it's a single dictionary, convert to a list
    mif_tile_list = [mif_tile_list]

# Convert to DataFrame
mif_tile_df = pd.DataFrame(mif_tile_list)

mif_tile_df['ScanRotationDeg'] = normalize_angle(float(mif_dict['MosaicInfo']['ReferenceInfo']['Beam']['ScanRot']))

# Apply the function to each row in the DataFrame
mif_tile_df['PixelSizeMicron'] = mif_tile_df['Filename'].apply(
    lambda fname: get_pixel_size_from_tif(fname, raw_data_folder))
# Apply function and expand the tuple into two columns
mif_tile_df[['ImageWidth', 'ImageHeight']] = mif_tile_df['Filename'].apply(
    lambda fname: get_image_size_from_tif(fname, raw_data_folder)
).to_list()
# Convert column to float
mif_tile_df['StageX'] = pd.to_numeric(mif_tile_df['StageX'], errors='coerce')
mif_tile_df['StageY'] = pd.to_numeric(mif_tile_df['StageY'], errors='coerce')

# Apply rotation to each row
mif_tile_df[['StageX_rot', 'StageY_rot']] = mif_tile_df.apply(
    lambda row: pd.Series(rotate_points((row['StageX'], row['StageY']), row['ScanRotationDeg'] * -1)),
    axis=1  # Apply function row-wise
)



min_stage_X = mif_tile_df['StageX_rot'].min()
min_stage_Y = mif_tile_df['StageY_rot'].min()

mif_tile_df['X0_micron'] = mif_tile_df['StageX_rot']-min_stage_X
mif_tile_df['Y0_micron'] = mif_tile_df['StageY_rot']-min_stage_Y
mif_tile_df['X0_pix'] = np.round(mif_tile_df['X0_micron']/mif_tile_df['PixelSizeMicron']).astype('uint')
mif_tile_df['Y0_pix'] = np.round(mif_tile_df['Y0_micron']/mif_tile_df['PixelSizeMicron']).astype('uint')
mif_tile_df

max_X_row = mif_tile_df.loc[mif_tile_df["X0_pix"].idxmax()]
max_Y_row = mif_tile_df.loc[mif_tile_df["Y0_pix"].idxmax()]
total_img_width = max_X_row['X0_pix'] + max_X_row['ImageWidth']
total_img_height = max_Y_row['Y0_pix'] + max_Y_row['ImageHeight']

print(f"Total image will be of size: {total_img_width}x{total_img_height}")
mif_tile_df

Total image will be of size: 7680x11360


,@row,@col,StartTime,TargetStageX,TargetStageY,StageX,StageY,WD,StigX,StigY,...,ScanRotationDeg,PixelSizeMicron,ImageWidth,ImageHeight,StageX_rot,StageY_rot,X0_micron,Y0_micron,X0_pix,Y0_pix
0,1,1,2024-10-14T15:15:38.349+02:00,-46629.9316976399,-50570.1953299105,-46629.941331,-50570.195253,0.00672752782702446,-0.295448243618011,0.463249564170837,...,-0.000031,0.100113,4000,4000,-46629.914396,-50570.220090,0.000000,736.835941,0,7360
1,1,2,2024-10-14T15:16:15.356+02:00,-46261.5140926594,-50570.1954411695,-46261.523361,-50570.195253,0.00672752782702446,-0.295448243618011,0.463249564170837,...,-0.000031,0.100113,4000,4000,-46261.496425,-50570.219894,368.417970,736.836137,3680,7360
2,2,2,2024-10-14T15:16:44.635+02:00,-46261.5142039185,-50938.61304615,-46261.523361,-50938.613224,0.00672752782702446,-0.295448243618011,0.463249564170837,...,-0.000031,0.100113,4000,4000,-46261.496229,-50938.637864,368.418166,368.418166,3680,3680
3,2,1,2024-10-14T15:17:27.589+02:00,-46629.9318088989,-50938.612934891,-46629.941331,-50938.613224,0.00672752782702446,-0.295448243618011,0.463249564170837,...,-0.000031,0.100113,4000,4000,-46629.914199,-50938.638060,0.000196,368.417970,0,3680
4,3,1,2024-10-14T15:18:09.966+02:00,-46629.9319201579,-51307.0305398714,-46629.941331,-51307.031194,0.00672752782702446,-0.295448243618011,0.463249564170837,...,-0.000031,0.100113,4000,4000,-46629.914003,-51307.056031,0.000392,0.000000,0,0
5,3,2,2024-10-14T15:18:41.970+02:00,-46261.5143151775,-51307.0306511304,-46261.523361,-51307.031194,0.00672752782702446,-0.295448243618011,0.463249564170837,...,-0.000031,0.100113,4000,4000,-46261.496033,-51307.055834,368.418363,0.000196,3680,0


In [19]:
def make_ome_pixel(img_path, sample_ome, position_x, position_y, pixel_size=1):
    
    img_path = Path(img_path)
    sample_ome = copy.deepcopy(sample_ome)
    pixel = sample_ome.images[0].pixels
    
    pixel.physical_size_x = pixel_size
    pixel.physical_size_y = pixel_size
    pixel.physical_size_x_unit = 'µm'
    pixel.physical_size_y_unit = 'µm'


    UUID = ome_types.model.TiffData.UUID(
        file_name=str(img_path.name),
        value=uuid.uuid4().urn
    )

    tiff_block = pixel.tiff_data_blocks[0]

    num_planes = tiff_block.plane_count
    tiff_block.uuid = UUID

    for i in range(num_planes):
        plane = ome_types.model.Plane(
            the_c=i, the_z=0, the_t=0,
            position_x=position_x, position_y=position_y,
            position_x_unit='µm',
            position_y_unit='µm',
        )
        pixel.planes.append(plane)
    
    return pixel


# Read image dtype from the first file (without loading data)
first_tif_path = raw_data_folder.joinpath(Path(mif_tile_df.iloc[0]['Filename']).name)

# generate a ome-xml template
tifffile.imwrite('sample.ome.tif', tifffile.imread(first_tif_path))
sample_ome = ome_types.from_tiff('sample.ome.tif')

pixel_list = []
for idx, row in mif_tile_df.iterrows():
    img_path = raw_data_folder.joinpath(Path(row['Filename']).name)
    x0 = row['X0_micron']
    y0 = row['Y0_micron']
    pixel_size = row['PixelSizeMicron']

    pixel_list.append(make_ome_pixel(img_path, sample_ome, x0, y0, pixel_size))

omexml = ome_types.model.OME()
omexml.images = [ome_types.model.Image(pixels=p) for p in pixel_list]

out_path = 'test.companion.ome'
print(f"Writing to {out_path}\n")
with open(out_path, 'w') as f:
    f.write(omexml.to_xml())

Writing to test.companion.ome

